In [1]:
import sys
sys.path.append("../")
import numpy as np
from utils.noise_generator import ColoredNoiseGenerator_Cholesky

def build_C_from_alpha(alpha, tgrid):
    N = len(tgrid)
    C = np.zeros((N, N), dtype=complex)
    for i in range(N):
        for j in range(N):
            C[i, j] = alpha(tgrid[j] - tgrid[i])
    return C

def rel_fro_error(A, B, eps=1e-15):
    return np.linalg.norm(A - B, 'fro') / (np.linalg.norm(B, 'fro') + eps)

def alpha(t):
    g = 2
    w = 0.5 + 2j
    a = lambda t: g * np.exp(-w * t)
    if t >= 0:
        return a(t)
    else:
        return np.conj(a(-t))

steps = 10
tmax = np.pi
dt = tmax / steps
tgrid = np.linspace(0, tmax, steps + 1)

C = build_C_from_alpha(alpha, tgrid)

noise_generator = ColoredNoiseGenerator_Cholesky(alpha, t_stop=tmax, N_steps=steps)

N_samples = 100000
C_hat = 0

for n in range(N_samples):
    z = noise_generator.sample_process()
    C_hat += np.outer(z, z.conj())

C_hat /= N_samples

print("Relative Frobenius error:", rel_fro_error(C_hat, C))

Relative Frobenius error: 0.005319302894117755


In [2]:
# bath correlation function in Cai_2020_CPAM
Delta = 1
beta = 5/Delta
wc = 2.5 * Delta
wmax = 4 * wc
CapL = 200
factor = 1 - np.exp(-wmax/wc)
wl = -wc * np.log(1 - np.linspace(1, CapL, CapL)/CapL * factor)
cl = wl * np.sqrt((0.2 * wc/CapL) * factor)

_coth = 1.0 / np.tanh(0.5 * beta * wl)
_pref = (cl**2) / (2.0 * wl)
def bath_corr(t):
    """
    Bath correlation function alpha(t).
    Supports scalar t or 1D array t (returns same shape).
    """
    # t_arr = np.asarray(t)
    # t1 = np.atleast_1d(t_arr).astype(float)
    # phase = wl[:, None] * t1[None, :]             # shape: (CapL, len(t))
    # out = np.sum(_pref[:, None] * (np.cos(phase) * _coth[:, None] - 1j * np.sin(phase)), axis=0)
    # return out[0] if t_arr.ndim == 0 else out
    res = 0
    for l in range(CapL):
        res += _pref[l] * (_coth[l] * np.cos(wl[l]*t) - 1j * np.sin(wl[l]*t))
    return res

tmax = 5/Delta
steps = 50
dt = tmax / steps
tgrid = np.linspace(0, tmax, steps + 1)


noise_generator = ColoredNoiseGenerator_Cholesky(bath_corr, t_stop=tmax, N_steps=steps)